## Loading Environment Variables

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

## Loading Gemini LLM

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
model="gemini-2.5-flash",
temperature=0
)

## Importing Tools in Notebook

In [3]:
import sys
sys.path.append('../src')
from tools import read_calendar, get_customer_profile

In [4]:
tools = {
    "read_calendar": read_calendar,
    "get_customer_profile": get_customer_profile
}

## Triage Node

In [5]:
def triage_node(state):
    email = state["email"]
    prompt = f"""
    Classify this email into one of:
    ignore
    notify_human
    respond
    Email:
    {email}
    Return only the label.
    """
    label = llm.invoke(prompt).content.strip().lower()
    return {**state, "triage": label}

## ReAct Agent (Reasoning + Tools)

In [6]:
def react_agent(state):
    email = state["email"]
    prompt = f"""
You are an email assistant.
You can use tools if needed.
Tools:
read_calendar
get_customer_profile
Email:
{email}
If you need a tool, write TOOL:<toolname>
Otherwise give reply.
"""
    response = llm.invoke(prompt).content
    if "TOOL:" in response:
        tool_name = response.replace("TOOL:", "").strip()
        tool_result = tools[tool_name]()
        return {**state, "response": tool_result}
    return {**state, "response": response}

## Build LangGraph Pipeline

In [7]:
from langgraph.graph import StateGraph
graph = StateGraph(dict)
graph.add_node("triage", triage_node)
graph.add_node("react", react_agent)
def route(state):
 if state["triage"] == "respond":
    return "react"
 else:
    return "end"
graph.add_conditional_edges("triage", route)
graph.set_entry_point("triage")
app = graph.compile()

## Load Email Dataset

In [8]:
import pandas as pd
emails = pd.read_csv("../data/sample_emails_with_triage_200.csv")

## Running Golden Test (25 Emails)

In [9]:
results = []
for _, row in emails.head(5).iterrows():
    email = row["body"]
    output = app.invoke({"email": email})
    results.append({
        "email": email,
        "triage": output["triage"],
        "response": output.get("response", "")
    })

Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.
Task triage with path ('__pregel_pull', 'triage') wrote to unknown channel branch:to:end, ignoring it.


## Saving Milestone-1 Output

In [10]:
pd.DataFrame(results).to_csv("../data/milestone1_output.csv", index=False)

## Accuracy Evaluation

In [12]:
gold = pd.read_csv("../data/golden_labels.csv")
pred = pd.read_csv("../data/milestone1_output.csv")
accuracy = (gold["expected"] == pred["triage"]).mean()
accuracy

np.float64(0.2)